# Random Forest

This notebook trains and evaluates the Random Forest using the best hyperparameters found in `main_M.ipynb` by 5-fold cross-validation.

**Project location assumption:** save this notebook as `imputer/randomforest.ipynb`, alongside `main_M.ipynb`. It reads `../data/data.csv` and imports `utils.py` from the project root.

In [18]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype

In [19]:
df = pd.read_csv("data/data.csv")

/tmp/ipykernel_6335/950551662.py:1: DtypeWarning: Columns (17: anzart_genossenschaft_janein, 23: aus_alarmanlage_janein, 51: aus_lastenaufzug_janein, 58: aus_pool_janein) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/data.csv")


In [20]:
def display_nan_stats(df):
    nan_stats = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_count": df.nunique(),
        "values": [
            "" if is_numeric_dtype(df[col])
            else df[col].dropna().unique().tolist()
            for col in df.columns
        ]
    })

    nan_stats = nan_stats.sort_values(
        "missing_percent",
        ascending=False
    )

    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", None
    ):
        display(nan_stats)

    return nan_stats

display_nan_stats(df)

,dtype,missing_count,missing_percent,unique_count,values
anzart_wohnen_auf_zeit_janein,float64,144778,100.00,0,
anzart_waz_moebliert_janein,float64,144778,100.00,0,
oadr_ortz_reg,float64,144778,100.00,0,
anz_parken_parkhaus,float64,144768,99.99,3,
aus_lastenaufzug_janein,object,144769,99.99,1,[True]
anzart_genossenschaft_janein,object,144677,99.93,1,[True]
aus_boden_terrakotta_janein,object,144547,99.84,1,[True]
anz_gebaeude,float64,144511,99.82,4,
anz_einheiten_gewerbe,float64,144447,99.77,11,
anz_einheiten_gewerbe_gesamt,float64,144362,99.71,21,


,dtype,missing_count,missing_percent,unique_count,values
anzart_wohnen_auf_zeit_janein,float64,144778,100.00,0,
anzart_waz_moebliert_janein,float64,144778,100.00,0,
oadr_ortz_reg,float64,144778,100.00,0,
anz_parken_parkhaus,float64,144768,99.99,3,
aus_lastenaufzug_janein,object,144769,99.99,1,[True]
...,...,...,...,...,...
objekttyp_fein,int64,0,0.00,13,
objekttyp_grob,int64,0,0.00,3,
zust_klassen5,int64,0,0.00,5,
startdate,str,0,0.00,1802,"[2022-11-16, 2024-10-07, 2024-10-09, 2024-10-1..."


In [21]:
def fill_missing_false_for_true_only(df):
    df = df.copy()
    processed_columns = []

    for col in df.columns:
        values = set(df[col].dropna().unique())

        if values == {True}:
            df[col] = df[col].fillna(False)
            processed_columns.append(col)

    print(f"Processed {len(processed_columns)} columns:")
    print(processed_columns)

    return df

df = fill_missing_false_for_true_only(df)
nan_stats = display_nan_stats(df)

Processed 51 columns:
['aadr_gewerblich_janein', 'anzart_genossenschaft_janein', 'anzart_indexmiete_janein', 'anzart_nachmieter_janein', 'aus_abstellraum_janein', 'aus_alarmanlage_janein', 'aus_bad_dusche_janein', 'aus_bad_gaestewc_janein', 'aus_bad_mit_fenster_janein', 'aus_bad_wanne_janein', 'aus_balkon_janein', 'aus_barrierefrei_janein', 'aus_boden_estrich_janein', 'aus_boden_fliesen_janein', 'aus_boden_kunststoff_janein', 'aus_boden_laminat_janein', 'aus_boden_linoleum_janein', 'aus_boden_marmor_janein', 'aus_boden_parkett_janein', 'aus_boden_teppich_janein', 'aus_boden_terrakotta_janein', 'aus_dachterrasse_janein', 'aus_garten_janein', 'aus_hauswirtschaftsraum_janein', 'aus_heiz_fernwaerme_janein', 'aus_heiz_fussboden_janein', 'aus_kamin_janein', 'aus_kueche_einbau_janein', 'aus_kueche_offen_janein', 'aus_lastenaufzug_janein', 'aus_lift_janein', 'aus_parken_garage_janein', 'aus_parken_janein', 'aus_parken_ladestation_janein', 'aus_parken_stellplaetze_janein', 'aus_parken_tiefgarag

,dtype,missing_count,missing_percent,unique_count,values
anzart_wohnen_auf_zeit_janein,float64,144778,100.00,0,
anzart_waz_moebliert_janein,float64,144778,100.00,0,
oadr_ortz_reg,float64,144778,100.00,0,
anz_parken_parkhaus,float64,144768,99.99,3,
anz_gebaeude,float64,144511,99.82,4,
anz_einheiten_gewerbe,float64,144447,99.77,11,
anz_einheiten_gewerbe_gesamt,float64,144362,99.71,21,
anz_parken_carports,float64,143186,98.90,4,
oeig_flachdach_janein,object,138458,95.63,2,"[False, True]"
oeig_freistehend_janein,object,137070,94.68,2,"[False, True]"


In [22]:
start_features = ['aus_boden_kunststoff_janein',
                  'aus_boden_laminat_janein',
                  'aus_boden_linoleum_janein',
                  'aus_boden_marmor_janein',
                  'aus_balkon_janein',
                  'aus_boden_estrich_janein', 
                  'aus_terrasse_janein',
                  'aus_boden_fliesen_janein',
                  'aus_boden_terrakotta_janein',
                  'aus_boden_parkett_janein',
                  'aus_heiz_fernwaerme_janein',
                  'aus_heiz_fussboden_janein',
                  'aus_kueche_einbau_janein',
                  'aus_kueche_offen_janein',
                  'oeig_neubau_janein',
                  'objekttyp_fein',
                  'aus_boden_teppich_janein',
                  'flaeche', # new features
                  'oeig_keller_janein',
                  'baujahr',
                  'etage',
                  'anz_zimmer',
                  'aus_klassen5',
                  'aadr_gewerblich_janein',
                  'aus_alarmanlage_janein',
                  'aus_abstellraum_janein',
                  'anzart_indexmiete_janein',
                  'anzart_nachmieter_janein',
                  'anzart_genossenschaft_janein',
                  'aus_bad_dusche_janein',
                  'aus_bad_mit_fenster_janein',
                  'aus_bad_gaestewc_janein',
                  'aus_barrierefrei_janein',
                  'aus_bad_wanne_janein',
                  'energie_niedrig_janein',
                  'aus_pool_janein',
                  'aus_sauna_janein',
                  'aus_parken_tiefgarage_janein',
                  'aus_hauswirtschaftsraum_janein',
                  'aus_garten_janein',
                  'aus_kamin_janein',
                  'aus_lastenaufzug_janein',
                  'aus_lift_janein',
                  'aus_parken_garage_janein',
                  'aus_parken_janein',
                  'aus_parken_ladestation_janein',
                  'oeig_loggia_janein',
                  'oeig_galerie_janein',
                  'zust_gepflegt_janein',
                  'zust_projektiert_janein',
                  'zust_sanierungsbedarf_janein',
                 ]

df = df[start_features + ['kosten_je_flaeche']].copy()

In [23]:
df = pd.get_dummies(
    df,
    columns=["objekttyp_fein", 'aus_klassen5'],
    dtype=int
) # one-hot encoding 
df = df.dropna(subset=df.select_dtypes(include="object").columns) # numeric NaN are imputed with EM later

In [24]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("boolean")

In [25]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import train_model, evaluate_model

In [26]:
target = "kosten_je_flaeche"

X = df.drop(columns=[target])
y = df[target]

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [28]:
from imputer.imputer import EMImputer, GMMImputer
num_cols = X_train.select_dtypes(include="float").columns # non-boolean features; target is not in X

imputer = GMMImputer(n_components=3).fit(X_train[num_cols]) # fit on train only
X_train[num_cols] = imputer.transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

print(list(num_cols), "EM iterations:", imputer.n_iter_)

/home/yeran/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but GMMImputer was fitted with feature names
  warnings.warn(


['flaeche', 'baujahr', 'etage', 'anz_zimmer'] EM iterations: 100


/home/yeran/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but GMMImputer was fitted with feature names
  warnings.warn(


## Best Random Forest parameters

These are the best parameters returned by the grid search you ran:

- `n_estimators=300`
- `min_samples_leaf=5`
- `max_features=1.0`
- `max_depth=None`
- `min_samples_split=2`
- `criterion="squared_error"`
- `ccp_alpha=0.0`

The recorded 5-fold CV results were RMSE ≈ 3883.76, MAE ≈ 2880.29, and R² ≈ 0.4408.

In [29]:
from sklearn.ensemble import RandomForestRegressor

best_params = {
    "n_estimators": 300,
    "criterion": "squared_error",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "ccp_alpha": 0.0,
    "bootstrap": True,
    "random_state": 1234,
    "n_jobs": 1,
}

best_model = RandomForestRegressor(**best_params)
best_model.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1234
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"

## Test-set evaluation

In [30]:
test_result = evaluate_model(
    best_model,
    X_test,
    y_test
)

test_result

{'rmse': np.float64(3839.0894101502713),
 'mae': 2830.5854930544624,
 'r2': 0.45613998241356757,
 'predictions': array([ 913.6005624 , 5109.30518853, 8000.72707943, ..., 7607.38451256,
         533.29075611,  799.99046153], shape=(23902,))}

## Optional: inspect feature importance

Feature importance is the Random Forest's impurity-based importance, not a causal effect.

In [31]:
feature_importance = (
    pd.Series(best_model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

feature_importance.head(20)

flaeche                       0.182639
aadr_gewerblich_janein        0.159773
baujahr                       0.121366
etage                         0.050970
zust_projektiert_janein       0.049672
anz_zimmer                    0.035569
anzart_indexmiete_janein      0.033040
aus_kueche_einbau_janein      0.027894
aus_lift_janein               0.016907
zust_gepflegt_janein          0.014837
aus_boden_fliesen_janein      0.014768
aus_bad_wanne_janein          0.013474
aus_heiz_fernwaerme_janein    0.013241
aus_boden_parkett_janein      0.013200
aus_garten_janein             0.012901
aus_heiz_fussboden_janein     0.012374
aus_balkon_janein             0.012322
aus_bad_dusche_janein         0.011935
aus_barrierefrei_janein       0.011406
aus_klassen5_2.0              0.010575
dtype: float64